# Simplicial complexes

A simplicial complex is a set of simplices — points, edges, triangles,
tetrahedra — that is **closed under faces**: if a simplex is in the complex,
every one of its sub-simplices is too. `qarp.graphs.SimplicialComplex`
enforces that on both sides, adding faces when you add a simplex and removing
cofaces when you remove one.


In [ ]:
from qarp import config
from qarp.graphs import SimplicialComplex

# The spring layout is seeded from here; without it figures move between runs.
config.seed = 7


## Closure when you add

Adding a triangle brings its three edges and three vertices with it.


In [ ]:
sc = SimplicialComplex([(0, 1, 2)])
sc.get_simplices()


## Drawing: simplices, not contours

Marks go by order: labelled points for vertices, line segments for edges,
plain filled polygons for triangles, and a hatched hull for a tetrahedron.

Only **facets** — the maximal simplices — are filled. A tetrahedron is
therefore one body rather than four separate triangular faces, and a missing
face shows up as an absent line rather than as a missing patch.


In [ ]:
# Orders 2 and 3 side by side: a plain triangle beside a hatched tetrahedron.
SimplicialComplex([(0, 1, 2), (2, 3, 4, 5), (0, 5)]).plot()


## Closure when you remove

`remove_simplex` takes the simplex **and every simplex containing it**. That
is what keeps the complex closed: a complex that kept `(0, 1, 2)` after
`(0, 1)` was removed would no longer be one.

Faces are left alone, so removing an edge does not remove its endpoints.


In [ ]:
sc = SimplicialComplex([(0,), (1,), (2,), (3,), (0, 1), (1, 2), (2, 3), (0, 1, 2)])
sc.add_simplex((0, 1, 3))
print("before:", sc.get_simplices())
sc.plot()


In [ ]:
# Removing the shared edge takes both triangles built on it.
sc.remove_simplex((0, 1))
print("after: ", sc.get_simplices())
sc.plot()


Note the blast radius: because cofaces go too, removing a **vertex**
removes every simplex it belongs to. That is the price of keeping the closure
invariant true at all times.


In [ ]:
sc2 = SimplicialComplex([(0, 1, 2), (2, 3)])
sc2.remove_simplex((0,))
sc2.get_simplices()


## The Euler characteristic

`chi = sum_k (-1)^k f_k` counts simplices by order with alternating sign. It is
a topological invariant, so it is a good way to check a complex is what you
think it is — a filled triangle is contractible (`chi = 1`), its hollow
boundary is a circle (`chi = 0`).


In [ ]:
filled = SimplicialComplex([(0, 1, 2)])
hollow = SimplicialComplex([(0, 1), (1, 2), (0, 2)])
print("filled triangle:", filled.f_vector(), "chi =", filled.euler_characteristic())
print("hollow triangle:", hollow.f_vector(), "chi =", hollow.euler_characteristic())

In [ ]:
hollow.plot()   # the 2-simplex is absent, so nothing is filled


## Above order 3, `plot` refuses

A 3-simplex already has no faithful 2D drawing — its hull marks where it is
rather than depicting it. Past that even the indicator stops meaning anything,
so `plot` raises instead of drawing something misleading. Summarise the complex
instead.


In [ ]:
from qarp.graphs import generate_complete_simplicial_complex

big = generate_complete_simplicial_complex(5)   # a 4-simplex
try:
    big.plot()
except ValueError as err:
    print(err)

print("max_dimension:", big.max_dimension(), " f-vector:", big.f_vector())
print("Euler characteristic:", big.euler_characteristic())


## Inspecting a complex

The complex is a container — `len`, iteration in `(order, vertices)` order, membership that
accepts a list in any vertex order — with accessors for its vertices, the neighbours of a
vertex, the cofaces of a simplex (every simplex containing it: exactly what `remove_simplex`
deletes), the $k$-skeleton and the 1-skeleton as a `qarp.graphs.Graph`.

In [ ]:
sc = SimplicialComplex([(0, 1, 2), (1, 2, 3), (3, 4)])
print(len(sc), "simplices;", (1, 2) in sc, [2, 1] in sc, (0, 3) in sc)
print("vertices:", sc.vertices())
print("neighbours of 2:", sc.neighbors(2))
print("cofaces of (1, 2):", sc.cofaces((1, 2)))
print("1-skeleton:", sc.skeleton(1).get_simplices())
print("as a graph:", list(sc.to_graph().edges))
sc.plot()

## Operators, not spectra

The boundary operators $\partial_k$ and the combinatorial Hodge Laplacians
$L_k = \partial_k^{\mathsf T}\partial_k + \partial_{k+1}\partial_{k+1}^{\mathsf T}$ come as
sparse integer matrices. Rows of $\partial_k$ index the $(k-1)$-simplices and columns the
$k$-simplices, both in `get_simplices` order, with sign $(-1)^i$ on the face that drops the
$i$-th vertex. $\partial_{k-1}\partial_k = 0$, and $L_0$ is the graph Laplacian of the
1-skeleton. Their spectra (Betti numbers, harmonic representatives) are deliberately left to
the caller.

In [ ]:
d1 = sc.boundary_matrix(1)
d2 = sc.boundary_matrix(2)
print("d1 (vertices x edges):\n", d1.toarray())
print("d2 (edges x triangles):\n", d2.toarray())
print("d1 @ d2 is zero:", (d1 @ d2).count_nonzero() == 0)
print("L0:\n", sc.hodge_laplacian(0).toarray())
print("L1:\n", sc.hodge_laplacian(1).toarray())